# 03.03 · Training an MLP end to end

**Chapter:** Deep Learning · **Milestone:** M1

## Goals
- [ ] Load a real dataset with torchvision; train/validation split
- [ ] Build an MLP with non-linear activations; understand why depth needs non-linearity
- [ ] Optimizers (SGD vs AdamW) and learning-rate schedules
- [ ] Evaluate, save and reload a checkpoint

## Resources
- [fast.ai – Practical Deep Learning for Coders](https://course.fast.ai/) — free
- [PyTorch – Optimizing model parameters](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) — free
- [d2l.ai – Multilayer perceptrons](https://d2l.ai/chapter_multilayer-perceptrons/index.html) — free

In [ ]:
import time

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

from aieng import DATA

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(0)

## 1. Data: Fashion-MNIST
70k grayscale 28×28 images of clothing in 10 classes. Harder than digits, small enough for a laptop.

In [ ]:
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.286,), (0.353,))])
train_full = datasets.FashionMNIST(DATA, train=True, download=True, transform=tfm)
test_set = datasets.FashionMNIST(DATA, train=False, download=True, transform=tfm)
train_set, val_set = random_split(train_full, [54_000, 6_000], generator=torch.Generator().manual_seed(0))
classes = train_full.classes

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = DataLoader(val_set, batch_size=1024)
test_loader = DataLoader(test_set, batch_size=1024)

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, (img, label) in zip(axes, train_set):
    ax.imshow(img[0], cmap="gray"); ax.set_title(classes[label], fontsize=8); ax.axis("off")
plt.show()

## 2. The model
Without activations, stacked linear layers collapse into one linear layer (a product of matrices is a matrix).
Non-linearities (ReLU, GELU — GELU is what transformers use) are what make depth useful.

In [ ]:
def make_mlp():
    return nn.Sequential(
        nn.Flatten(),  # (B, 1, 28, 28) -> (B, 784)
        nn.Linear(784, 512), nn.GELU(),
        nn.Linear(512, 256), nn.GELU(),
        nn.Linear(256, 10),  # logits; CrossEntropyLoss applies softmax internally
    ).to(device)

print(make_mlp(), "\nparameters:", sum(p.numel() for p in make_mlp().parameters()))

## 3. Train / evaluate functions

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    loss_sum = 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss_sum += nn.functional.cross_entropy(logits, yb, reduction="sum").item()
            correct += (logits.argmax(1) == yb).sum().item()
            total += len(yb)
    return loss_sum / total, correct / total


def train(model, opt, epochs=3, scheduler=None):
    history = []
    for epoch in range(epochs):
        model.train()
        start = time.perf_counter()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = nn.functional.cross_entropy(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            if scheduler:
                scheduler.step()  # per-batch schedule
        val_loss, val_acc = evaluate(model, val_loader)
        history.append((loss.item(), val_loss, val_acc))
        print(f"epoch {epoch}: train loss {loss.item():.3f} | val loss {val_loss:.3f} | val acc {val_acc:.3f} | {time.perf_counter() - start:.1f}s")
    return history

## 4. SGD vs AdamW
- **SGD**: `p -= lr * grad`. Simple, needs tuning, often + momentum.
- **Adam/AdamW**: per-parameter adaptive step sizes from running averages of gradients and their squares.
  AdamW = Adam with decoupled weight decay. **The default optimiser for transformers.**

In [ ]:
EPOCHS = 3
torch.manual_seed(0); sgd_model = make_mlp()
print("SGD"); h_sgd = train(sgd_model, torch.optim.SGD(sgd_model.parameters(), lr=0.05), EPOCHS)
torch.manual_seed(0); adam_model = make_mlp()
print("AdamW"); h_adam = train(adam_model, torch.optim.AdamW(adam_model.parameters(), lr=1e-3, weight_decay=0.01), EPOCHS)

## 5. Learning-rate schedules
Warm up from a small LR, then decay. **One-cycle** / **cosine with warmup** is what most LLM training uses.

In [ ]:
torch.manual_seed(0); sched_model = make_mlp()
opt = torch.optim.AdamW(sched_model.parameters(), lr=3e-3, weight_decay=0.01)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=EPOCHS * len(train_loader))
print("AdamW + OneCycle"); h_sched = train(sched_model, opt, EPOCHS, sched)

for name, h in [("SGD", h_sgd), ("AdamW", h_adam), ("AdamW+OneCycle", h_sched)]:
    plt.plot([v[2] for v in h], "o-", label=name)
plt.xlabel("epoch"); plt.ylabel("val accuracy"); plt.legend(); plt.show()

## 6. Test once, inspect errors, save a checkpoint

In [ ]:
best = max([sgd_model, adam_model, sched_model], key=lambda m: evaluate(m, val_loader)[1])
test_loss, test_acc = evaluate(best, test_loader)
print(f"test accuracy: {test_acc:.3f}")

from sklearn.metrics import ConfusionMatrixDisplay

preds, labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        preds.append(best(xb.to(device)).argmax(1).cpu()); labels.append(yb)
ConfusionMatrixDisplay.from_predictions(torch.cat(labels), torch.cat(preds), display_labels=classes, xticks_rotation=60)
plt.show()

ckpt = DATA / "fashion_mlp.pt"
torch.save(best.state_dict(), ckpt)  # save weights only (portable); not the whole object
reloaded = make_mlp(); reloaded.load_state_dict(torch.load(ckpt, map_location=device))
print("reloaded test acc:", round(evaluate(reloaded, test_loader)[1], 3))

## Exercises
1. Reach **> 88% test accuracy** (more epochs, wider layers, dropout from 03.05, tuned LR). Only touch the test set at the end.
2. Plot train loss **per batch** for SGD vs AdamW on the same axes (log scale).
3. Which classes get confused most? Look at 10 misclassified images. Would a human get them right?

In [ ]:
# your code here